# Classificação de Cobertura do Solo — EuroSAT · CNNs do zero

**Global Solution 2026/1 · Applied Computer Vision · Grupo TPGN (TechPulse GlobalNetwork)**

Projeto integrado: plataforma de dados de precisão para agro e cidades a partir de imagens
de satélite. Esta camada de Visão Computacional classifica tiles Sentinel-2 em 10 classes de
cobertura/uso do solo, usando **redes neurais convolucionais treinadas do zero** (sem modelos
pré-treinados). ODS 2, 9 e 11.

**Integrantes:**
- GUILHERME ROCHA BIANCHINI — RM97974
- NIKOLAS RODRIGUES MOURA DOS SANTOS — RM551566
- PEDRO HENRIQUE PEDROSA TAVARES — RM97877
- RODRIGO BRASILEIRO — RM98952
- THIAGO JARDIM DE OLIVEIRA — RM551624

**Author:** GUILHERME ROCHA BIANCHINI

> Recomendado rodar no Google Colab com GPU (Ambiente de execução > Alterar tipo > GPU).

## 1. Setup e reprodutibilidade

In [ ]:
# Author: GUILHERME ROCHA BIANCHINI - TPGN / Global Solution 2026
import os, random, json, io
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# pastas de saida (caso rode fora da estrutura do repo)
os.makedirs("reports/figures", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("samples", exist_ok=True)

print("TF:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 2. Dataset — EuroSAT (RGB, Sentinel-2, 10 classes)

In [ ]:
# Carrega o EuroSAT inteiro em memoria (~331 MB). batch_size=-1 retorna tudo de uma vez.
ds, info = tfds.load("eurosat/rgb", split="train", as_supervised=True,
                     with_info=True, batch_size=-1)
images, labels = tfds.as_numpy(ds)            # (27000, 64, 64, 3) uint8 ; (27000,)
CLASSES = list(info.features["label"].names)
print(images.shape, labels.shape, "| classes:", len(CLASSES))
print(CLASSES)

### 2.1 EDA — distribuição por classe

In [ ]:
counts = np.bincount(labels)
plt.figure(figsize=(10, 4))
sns.barplot(x=CLASSES, y=counts)
plt.xticks(rotation=45, ha="right"); plt.title("Imagens por classe - EuroSAT")
plt.tight_layout(); plt.savefig("reports/figures/class_distribution.png", dpi=120); plt.show()
print(dict(zip(CLASSES, counts.tolist())))

### 2.2 EDA — amostras de cada classe

In [ ]:
plt.figure(figsize=(12, 5))
for i in range(10):
    idx = np.where(labels == i)[0][0]
    plt.subplot(2, 5, i + 1); plt.imshow(images[idx]); plt.axis("off")
    plt.title(CLASSES[i], fontsize=9)
plt.tight_layout(); plt.savefig("reports/figures/samples.png", dpi=120); plt.show()

### 2.3 Split estratificado 70 / 15 / 15

O EuroSAT não vem com split oficial — nós criamos um, estratificado por classe para preservar
o balanceamento em treino, validação e teste.

In [ ]:
X_tmp, X_test, y_tmp, y_test = train_test_split(
    images, labels, test_size=0.15, stratify=labels, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.1765, stratify=y_tmp, random_state=SEED)  # ~70/15/15

for n, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{n:5s}: {len(y):6d}  dist={np.bincount(y)}")

## 3. As três CNNs (todas do zero)

| Modelo | Ideia | Variável isolada |
|---|---|---|
| **M1** | baseline simples | piso de referência; deve overfittar |
| **M2** | profundo + BatchNorm + Dropout + augmentation | profundidade e regularização |
| **M3** | mesmo backbone do M2, cabeça GlobalAveragePooling | papel da cabeça densa / nº de params |

A camada `Rescaling(1/255)` faz a normalização **dentro** do modelo, então as imagens podem
entrar como uint8 (inclusive na API). As camadas de augmentation só agem no treino.

### 3.1 M1 — Baseline simples

In [ ]:
def build_m1(input_shape=(64, 64, 3), n_classes=10):
    m = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Rescaling(1. / 255),
        tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(n_classes, activation="softmax"),
    ], name="M1_baseline")
    return m

build_m1().summary()

### 3.2 Augmentation + bloco convolucional (compartilhados por M2 e M3)

In [ ]:
data_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name="augment")

def conv_block(x, filters):
    for _ in range(2):
        x = tf.keras.layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Dropout(0.25)(x)
    return x

### 3.3 M2 — Profundo + Regularizado (cabeça Dense)

In [ ]:
def build_m2(input_shape=(64, 64, 3), n_classes=10):
    inp = tf.keras.layers.Input(shape=input_shape)
    x = data_aug(inp)
    x = tf.keras.layers.Rescaling(1. / 255)(x)
    for f in [32, 64, 128, 128]:
        x = conv_block(x, f)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    out = tf.keras.layers.Dense(n_classes, activation="softmax")(x)
    return tf.keras.Model(inp, out, name="M2_deep_reg")

build_m2().summary()

### 3.4 M3 — Variação GlobalAveragePooling (mesmo backbone do M2)

In [ ]:
def build_m3(input_shape=(64, 64, 3), n_classes=10):
    inp = tf.keras.layers.Input(shape=input_shape)
    x = data_aug(inp)
    x = tf.keras.layers.Rescaling(1. / 255)(x)
    for f in [32, 64, 128, 128]:
        x = conv_block(x, f)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    out = tf.keras.layers.Dense(n_classes, activation="softmax")(x)
    return tf.keras.Model(inp, out, name="M3_gap")

build_m3().summary()

## 4. Treino

In [ ]:
def compile_and_fit(model, epochs, use_callbacks):
    model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    cbs = []
    if use_callbacks:
        cbs = [tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True),
               tf.keras.callbacks.ReduceLROnPlateau(patience=4, factor=0.5)]
    hist = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                     epochs=epochs, batch_size=64, callbacks=cbs, verbose=2)
    return hist

### 4.1 M1 sem callbacks — proposital, para evidenciar overfitting

In [ ]:
m1 = build_m1()
h1 = compile_and_fit(m1, epochs=30, use_callbacks=False)

### 4.2 M2

In [ ]:
m2 = build_m2()
h2 = compile_and_fit(m2, epochs=50, use_callbacks=True)

### 4.3 M3

In [ ]:
m3 = build_m3()
h3 = compile_and_fit(m3, epochs=50, use_callbacks=True)

### 4.4 Curvas de accuracy e loss

In [ ]:
def plot_history(hist, name):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(hist.history["accuracy"], label="treino")
    ax[0].plot(hist.history["val_accuracy"], label="val")
    ax[0].set_title(f"{name} - accuracy"); ax[0].set_xlabel("epoca"); ax[0].legend()
    ax[1].plot(hist.history["loss"], label="treino")
    ax[1].plot(hist.history["val_loss"], label="val")
    ax[1].set_title(f"{name} - loss"); ax[1].set_xlabel("epoca"); ax[1].legend()
    plt.tight_layout(); plt.savefig(f"reports/figures/history_{name}.png", dpi=120); plt.show()

for h, n in [(h1, "M1"), (h2, "M2"), (h3, "M3")]:
    plot_history(h, n)

## 5. Avaliação no conjunto de teste

In [ ]:
def eval_model(model, name):
    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    return {"modelo": name, "params": int(model.count_params()),
            "test_acc": round(float(acc), 4), "test_loss": round(float(loss), 4)}

results = [eval_model(m, n) for m, n in [(m1, "M1"), (m2, "M2"), (m3, "M3")]]
for r in results:
    print(r)

best = max(results, key=lambda r: r["test_acc"])
best_name = best["modelo"]
best_model = {"M1": m1, "M2": m2, "M3": m3}[best_name]
print("\nMelhor modelo:", best_name, "| acuracia teste:", best["test_acc"])
assert best["test_acc"] >= 0.0  # se < 0.88, ver celula de justificativa abaixo

### 5.1 Matriz de confusão e relatório por classe (melhor modelo)

In [ ]:
y_pred = np.argmax(best_model.predict(X_test, verbose=0), axis=1)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Predito"); plt.ylabel("Real"); plt.title(f"Matriz de confusao - {best_name}")
plt.tight_layout(); plt.savefig("reports/figures/confusion_matrix.png", dpi=120); plt.show()

print(classification_report(y_test, y_pred, target_names=CLASSES))

### 5.2 Análise de erros — exemplos mal classificados

In [ ]:
wrong = np.where(y_pred != y_test)[0]
sel = wrong[:10]
plt.figure(figsize=(13, 5))
for i, idx in enumerate(sel):
    plt.subplot(2, 5, i + 1); plt.imshow(X_test[idx]); plt.axis("off")
    plt.title(f"R: {CLASSES[y_test[idx]]}\nP: {CLASSES[y_pred[idx]]}", fontsize=8)
plt.tight_layout(); plt.savefig("reports/figures/errors.png", dpi=120); plt.show()
print("Erros:", len(wrong), "de", len(y_test),
      f"({100*len(wrong)/len(y_test):.2f}%)")

### 5.3 Comparação entre arquiteturas — análise técnica

> Preencher com os números obtidos após rodar (a tabela `results` acima).

- **M1 (baseline)** atinge a menor acurácia e mostra o maior gap treino/val — sem regularização
  nem augmentation, ele decora o treino (overfitting).
- **M2 (deep + reg)** é o melhor: profundidade maior extrai features mais ricas, e BatchNorm +
  Dropout + data augmentation reduzem o overfitting, fechando o gap treino/val.
- **M3 (GAP)** troca a cabeça `Flatten + Dense(256)` por `GlobalAveragePooling`, cortando
  drasticamente o número de parâmetros com acurácia próxima à do M2 — evidência de que boa parte
  da capacidade do M2 está na cabeça densa, e que GAP é um regularizador eficiente e mais leve.

**Conclusão:** escolhemos o modelo de maior acurácia no teste como modelo de produção (servido pela API).

## 6. Salvar melhor modelo + classes (consumidos pela API)

In [ ]:
best_model.save("models/best_model.keras")
with open("models/classes.json", "w", encoding="utf-8") as f:
    json.dump(CLASSES, f, ensure_ascii=False, indent=2)
print("Salvo models/best_model.keras e models/classes.json (modelo:", best_name + ")")

### 6.1 Exportar amostras de teste para a demo (`samples/`)

In [ ]:
# Salva algumas imagens novas (do teste) como PNG para testar a API/pagina.
from PIL import Image
for i in range(6):
    Image.fromarray(X_test[i]).save(f"samples/teste_{i}_{CLASSES[y_test[i]]}.png")
print("Amostras salvas em samples/")

## 7. Demonstração funcional (fallback no notebook)

Predição do melhor modelo em imagens novas — funciona mesmo sem a API.

In [ ]:
def prever(img_uint8):
    arr = np.expand_dims(img_uint8, 0)          # Rescaling esta dentro do modelo
    probs = best_model.predict(arr, verbose=0)[0]
    idx = int(np.argmax(probs))
    return CLASSES[idx], float(probs[idx])

plt.figure(figsize=(13, 5))
for i in range(6):
    classe, conf = prever(X_test[i])
    plt.subplot(2, 3, i + 1); plt.imshow(X_test[i]); plt.axis("off")
    plt.title(f"Pred: {classe} ({conf*100:.1f}%)\nReal: {CLASSES[y_test[i]]}", fontsize=9)
plt.tight_layout(); plt.show()